In [ ]:
import pandas as pd

bom_file = r"D:/Tushar/main_with_subs_only.xlsx"
comparison_file = r"D:/Tushar/Comparison data.xlsx"

# Read files
bom_df = pd.read_excel(bom_file)
comp_df = pd.read_excel(comparison_file)

bom_df.columns = bom_df.columns.str.strip()
comp_df.columns = comp_df.columns.str.strip()

def normalize(series):
    return series.astype(str).str.strip().str.upper()

# Normalize keys
bom_df['Sub_Label'] = normalize(bom_df['Sub_Label'])
bom_df['Main_Label'] = normalize(bom_df['Main_Label'])
comp_df['Material'] = normalize(comp_df['Material'])

# Prepare output
child_list = bom_df['Main_Label'].unique()
output_df = pd.DataFrame({'Material': child_list})

# Loop through columns exactly as they appear
for col in comp_df.columns:

    if col == 'Material':
        continue

    print(f"Processing: {col}")

    comp_df['Demand'] = pd.to_numeric(comp_df[col], errors='coerce').fillna(0)
    lookup = dict(zip(comp_df['Material'], comp_df['Demand']))

    # === EXACT SAME LOGIC START ===
    results = []
    current_child = None
    running_total = 0

    for _, row in bom_df.iterrows():

        child = row['Main_Label']
        switch = row['Sub_Label']
        usage = pd.to_numeric(row['Sub_Count'], errors='coerce') or 0

        daily_switch = lookup.get(switch, 0)
        contribution = daily_switch * usage

        if current_child is None:
            current_child = child

        if child != current_child:
            results.append((current_child, running_total))
            current_child = child
            running_total = 0

        running_total += contribution

    if current_child is not None:
        results.append((current_child, running_total))
    # === EXACT SAME LOGIC END ===

    col_df = pd.DataFrame(results, columns=['Material', col])

    output_df = output_df.merge(col_df, on='Material', how='left')

output_df = output_df.fillna(0)

output_df.to_excel("Child_From_Comparison_SameLogic.xlsx", index=False)

print("Saved output file.")


In [ ]:
import pandas as pd
import re

# ===============================
# LOAD FILE
# ===============================

file_path = "child_combined_file.xlsx"   # change if needed
df = pd.read_excel(file_path)

# Fill missing values
df = df.fillna(0)

# ===============================
# EXTRACT ALL DATES
# ===============================

columns = df.columns.tolist()

dates = sorted(
    list(set(re.findall(r"\d{4}-\d{2}-\d{2}", " ".join(columns))))
)

# ===============================
# DAILY DEVIATION
# ===============================

for date in dates:
    
    indent_col = f"{date} Indent"
    actual_col = f"{date} Total Production Plan"
    
    if indent_col in df.columns and actual_col in df.columns:
        
        # Daily deviation
        df[f"{date} Deviation"] = df[actual_col] - df[indent_col]
        
        # Percentage deviation (safe divide)
        df[f"{date} Deviation_%"] = (
            df[f"{date} Deviation"] /
            df[indent_col].replace(0, pd.NA)
        ) * 100
        
        # Status
        df[f"{date} Status"] = df[f"{date} Deviation"].apply(
            lambda x: "Over" if x > 0 else ("Short" if x < 0 else "OK")
        )

# ===============================
# MONTH TILL DATE SUMMARY
# ===============================

indent_cols = [
    f"{d} Indent"
    for d in dates
    if f"{d} Indent" in df.columns
]

actual_cols = [
    f"{d} Total Production Plan"
    for d in dates
    if f"{d} Total Production Plan" in df.columns
]

df["Total_Indent_Till_Date"] = df[indent_cols].sum(axis=1)
df["Total_Actual_Till_Date"] = df[actual_cols].sum(axis=1)

df["Total_Deviation"] = (
    df["Total_Actual_Till_Date"] - df["Total_Indent_Till_Date"]
)

df["Total_Deviation_%"] = (
    df["Total_Deviation"] /
    df["Total_Indent_Till_Date"].replace(0, pd.NA)
) * 100

# ===============================
# TOTAL STATUS + SHORTAGE RISK
# ===============================

df["Total Status"] = df["Total_Deviation"].apply(
    lambda x: "Over" if x > 0 else ("Short" if x < 0 else "OK")
)

df["Shortage Risk"] = df["Total_Deviation"].apply(
    lambda x: "⚠️ Risk" if x < 0 else "Safe"
)

# ===============================
# SAVE OUTPUT
# ===============================

output_file = "child_indent_vs_actual_analysis.xlsx"
df.to_excel(output_file, index=False)

print("Child part deviation analysis completed successfully.")
